In [ ]:
from google.colab import drive
drive.mount('/content/drive')

NLP_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP-Costumer-Classifier/'
print("Drive connected!")
print(f"Path: {NLP_PATH}")

In [ ]:
!pip install nltk -q

import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

print("NLTK ready!")

In [ ]:
import pandas as pd
import numpy as np
import re
import pickle
import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

print("All libraries imported!")

In [ ]:
df = pd.read_csv(NLP_PATH + 'consumer_complaints.csv', low_memory=False)

# Keep only what we need
df = df[['narrative', 'product']].copy()
df.columns = ['text', 'category']

# Drop empty rows
df = df.dropna(subset=['text', 'category'])
df = df[df['text'].str.strip() != '']

print(f"Shape: {df.shape}")
print(f"\nUnique categories BEFORE mapping:")
print(df['category'].unique())

In [ ]:
# Map Categories
# Check exact values first then map
print("Raw category values:")
print(df['category'].value_counts())

In [ ]:
# Map all possible formats — covers both underscore and space versions
category_map = {
    # Space versions
    'Mortgage':                          'Mortgage',
    'Debt collection':                   'Debt Collection',
    'Credit reporting':                  'Credit Reporting',
    'Credit card':                       'Credit Card',
    'Bank account or service':           'Bank Account',
    'Student loan':                      'Student Loan',
    'Consumer Loan':                     'Consumer Loan',
    'Credit reporting, credit repair services, or other personal consumer reports': 'Credit Reporting',
    'Credit card or prepaid card':       'Credit Card',
    'Checking or savings account':       'Bank Account',
    'Vehicle loan or lease':             'Vehicle Loan',
    'Payday loan, title loan, or personal loan': 'Personal Loan',
    'Money transfer, virtual currency, or money service': 'Money Transfer',
    # Underscore versions (your dataset uses these)
    'credit_card':                       'Credit Card',
    'debt_collection':                   'Debt Collection',
    'mortgage':                          'Mortgage',
    'credit_reporting':                  'Credit Reporting',
    'bank_account_or_service':           'Bank Account',
    'student_loan':                      'Student Loan',
    'consumer_loan':                     'Consumer Loan',
    'vehicle_loan':                      'Vehicle Loan',
    'personal_loan':                     'Personal Loan',
    'money_transfer':                    'Money Transfer',
    'payday_loan':                       'Personal Loan',
    'prepaid_card':                      'Credit Card',
    'checking_or_savings_account':       'Bank Account',
    'other_financial_service':           'Other',
    'money_transfers':                   'Money Transfer',
}

df['category'] = df['category'].map(category_map)
df = df.dropna(subset=['category'])

print(f"\nRows after mapping: {len(df):,}")
print("\nSimplified categories:")
print(df['category'].value_counts())

In [ ]:
# Clean Text Function
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Convert to string (safety)
    text = str(text)
    # Lowercase
    text = text.lower()
    # Remove numbers, punctuation, special chars
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Remove stopwords
    words = text.split()
    words = [w for w in words if w not in stop_words]
    # Lemmatize
    words = [lemmatizer.lemmatize(w) for w in words]
    return ' '.join(words)

print(f"Cleaning {len(df):,} rows...")
print("Wait 5-10 minutes...")

df['clean_text'] = df['text'].apply(clean_text)

print(f"\n✅ Text cleaning done!")
print(f"Total rows: {len(df):,}")
print(f"\nOriginal sample:")
print(df['text'].iloc[0][:200])
print(f"\nCleaned sample:")
print(df['clean_text'].iloc[0][:200])

In [ ]:
# Encode Categories
le = LabelEncoder()
df['label'] = le.fit_transform(df['category'])

print("Category → Number mapping:")
for i, cat in enumerate(le.classes_):
    print(f"  {i} → {cat}")

In [ ]:
# Sample 50K Rows
df_sample = df.sample(min(50000, len(df)), random_state=42)

print(f"Using {len(df_sample):,} samples")
print(f"\nCategory distribution:")
print(df_sample['category'].value_counts())

In [ ]:
# Train Test Split
X = df_sample['clean_text']
y = df_sample['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training samples: {len(X_train):,}")
print(f"Testing samples:  {len(X_test):,}")

In [ ]:
# Save to Drive
with open(NLP_PATH + 'nlp_data.pkl', 'wb') as f:
    pickle.dump({
        'X_train': X_train,
        'X_test':  X_test,
        'y_train': y_train,
        'y_test':  y_test,
        'df':      df_sample,
        'le':      le
    }, f)

print("="*50)
print("✅ NOTEBOOK 2 COMPLETE!")
print("="*50)
print("Saved: nlp_data.pkl to Drive")
print("\nNow open Notebook 3 — Classification Models")